# Embedding Model Fine-tuning with Sentence Transformers

[![Open In Colab](https://img.shields.io/badge/Open%20In-Colab-blue?style=for-the-badge&logo=google-colab)](https://colab.research.google.com/github/dnth/rag-datakit/blob/main/nbs/02_train.ipynb)
[![Open In Kaggle](https://img.shields.io/badge/Open%20In-Kaggle-blue?style=for-the-badge&logo=kaggle)](https://kaggle.com/kernels/welcome?src=https://github.com/dnth/rag-datakit/blob/main/nbs/02_train.ipynb)

This notebook demonstrates how to fine-tune an embedding model using the synthetic triplet data generated from Singapore SkillsFuture Framework job descriptions. We'll use the sentence-transformers library to train a model that can better understand job-related semantic similarity for improved retrieval and matching.

## What you'll learn:
- How to set up sentence-transformers training pipeline
- Configuring training arguments for embedding models
- Using MultipleNegativesRankingLoss for triplet training
- Monitoring training with Weights & Biases
- Saving and publishing trained models

## Installation

Install the rag-datakit package which includes all necessary dependencies including distilabel, transformers, and dataset utilities. Uncomment the cell below to install if you haven't already.

On Google Colab you might need to uninstall the existing packages due to conflicting versions.

In [ ]:
# !pip uninstall -y transformers torch torchvision

In [ ]:
# !pip install git+https://github.com/dnth/rag-datakit.git

## Import Required Libraries

We begin by importing the essential libraries for training our embedding model:

- **sentence_transformers**: The core library providing tools for training and evaluating sentence transformers
- **datasets**: Hugging Face's library for loading and managing our training dataset
- **wandb**: Weights & Biases for experiment tracking and visualization of training metrics

In [1]:
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import BatchSamplers
from datasets import load_dataset

## Dataset Loading and Inspection

We load the preprocessed training dataset with train/validation splits that were created in the previous notebook. This dataset contains triplets specifically formatted for embedding model training using contrastive learning approaches.

**Dataset source**: `dnth/ssf-train-valid`  
**Structure**:
- **anchor**: Original job descriptions from the SkillsFuture Framework
- **positive**: Semantically similar paraphrases generated through synthetic data techniques
- **negative**: Semantically different job descriptions used as contrastive examples

Let's examine the dataset structure and review a sample triplet to understand the data format:

In [2]:
dataset = load_dataset("dnth/ssf-train-valid")
dataset

README.md:   0%|          | 0.00/484 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/2.98M [00:00<?, ?B/s]

data/valid-00000-of-00001.parquet:   0%|          | 0.00/752k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4180 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/1045 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 4180
    })
    valid: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 1045
    })
})

In [3]:
dataset['valid'][0]

{'anchor': 'The Logistics Solutions and Implementation Director/Tailored Supply Chain Director/Channel Operations Director is responsible for managing the processes of business development and implementing custom-made or tailored end-to-end complex logistics solutions for customers, including managing post implementation optimisation. He/She is also responsible for managing logistics solutioning business resources. Resourceful and persuasive, he is required to manage resources and obtain buy-in from internal and external stakeholders. He is also expected to lead a department and make business decisions independently.',
 'positive': 'Logistics Solutions Director overseeing business development and tailored logistics implementations, focusing on end-to-end solutions and post-implementation optimization while managing resources and stakeholder engagement.',
 'negative': 'Junior Risk Management Analyst responsible for evaluating and mitigating risks within the financial services sector, co

## Initialize Weights & Biases and Model Configuration

We initialize Weights & Biases for experiment tracking and define our base model and save path. We're using the `all-minilm-l6-v2` model as our starting point, which is a efficient, general-purpose sentence embedding model that provides a good balance between speed and performance.

In [4]:
import wandb

model_id = "nomic-ai/modernbert-embed-base"
save_model_path = "./models/cxs-modernbert-embed-base"

# wandb.login()
wandb.init(project="rag-datakit-finetunes", name="cxs-modernbert-embed-base")

wandb: Currently logged in as: dnth to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Configure Training Arguments

We configure the training arguments for our embedding model. These parameters control various aspects of the training process including batch sizes, learning rate, precision settings, and checkpointing behavior. Key configurations include:

- 5 training epochs with cosine learning rate scheduler
- Mixed precision training using bf16 for efficiency
- Gradient accumulation to achieve an effective batch size of 512
- NO_DUPLICATES batch sampler to ensure diverse negative samples
- Checkpointing at the end of each epoch with a limit of 3 saved models
- Evaluation after each epoch to monitor validation loss

In [5]:
args = SentenceTransformerTrainingArguments(
    output_dir=save_model_path,
    num_train_epochs=5,                         # number of epochs
    per_device_train_batch_size=32,             # train batch size
    gradient_accumulation_steps=16,             # for a global batch size of 512
    per_device_eval_batch_size=16,              # evaluation batch size
    warmup_ratio=0.1,                           # warmup ratio
    learning_rate=2e-5,                         # learning rate, 2e-5 is a good value
    lr_scheduler_type="cosine",                 # use cosine learning rate scheduler
    optim="adamw_torch_fused",                  # use fused adamw optimizer
    tf32=False,                                  # use tf32 precision
    bf16=True,                                  # use bf16 precision
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # MultipleNegativesRankingLoss benefits from no duplicate samples in a batch
    eval_strategy="epoch",                      # evaluate after each epoch
    save_strategy="epoch",                      # save after each epoch
    logging_strategy="epoch",                   # log after each epoch
    save_total_limit=3,                         # save only the last 3 models
    load_best_model_at_end=True,                # load the best model when training ends
    report_to="wandb"
    )

## Initialize Model, Loss Function, and Trainer

We initialize our sentence transformer model and configure the training components:

- Load the base `all-minilm-l6-v2` model from sentence-transformers
- Configure `MultipleNegativesRankingLoss` which is ideal for triplet training as it maximizes the similarity between anchor and positive pairs while minimizing similarity between anchor and negative pairs
- Set up the `SentenceTransformerTrainer` with our model, training arguments, datasets, and loss function

In [6]:
model = SentenceTransformer(model_id)
train_loss = MultipleNegativesRankingLoss(model)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['valid'],  
    loss=train_loss,
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

## Execute Model Training

We start the training process using our configured trainer. The model will train for 5 epochs, with evaluation happening after each epoch. The training progress and metrics are tracked through Weights & Biases, showing both training and validation loss metrics.

As training progresses, we can observe the validation loss decreasing, indicating that our model is learning to distinguish between semantically similar and dissimilar job descriptions.

In [7]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.216200,0.013254
2,0.019500,0.009457
3,0.013600,0.007958
4,0.011500,0.007427
5,0.011200,0.007377


TrainOutput(global_step=45, training_loss=0.054387802547878686, metrics={'train_runtime': 312.3696, 'train_samples_per_second': 66.908, 'train_steps_per_second': 0.144, 'total_flos': 0.0, 'train_loss': 0.054387802547878686, 'epoch': 5.0})

## Save and Upload Trained Model

After training is complete, we save the fine-tuned model to disk and upload it to Weights & Biases for versioning and sharing. The saved model includes all necessary components:

- Model weights and configuration
- Tokenizer files
- Pooling layer configuration
- Normalization components

This ensures that the model can be easily loaded and used for inference later.

In [8]:
trainer.save_model()

In [9]:
import os
wandb.save(os.path.join(save_model_path, "*"))

wandb: WARNING Symlinked 15 files into the W&B run directory, call wandb.save again to sync new files.


['/home/dnth/Desktop/rag-datakit/nbs/wandb/run-20250908_143209-7d4hqku2/files/models/cxs-modernbert-embed-base/special_tokens_map.json',
 '/home/dnth/Desktop/rag-datakit/nbs/wandb/run-20250908_143209-7d4hqku2/files/models/cxs-modernbert-embed-base/2_Normalize',
 '/home/dnth/Desktop/rag-datakit/nbs/wandb/run-20250908_143209-7d4hqku2/files/models/cxs-modernbert-embed-base/1_Pooling',
 '/home/dnth/Desktop/rag-datakit/nbs/wandb/run-20250908_143209-7d4hqku2/files/models/cxs-modernbert-embed-base/config.json',
 '/home/dnth/Desktop/rag-datakit/nbs/wandb/run-20250908_143209-7d4hqku2/files/models/cxs-modernbert-embed-base/training_args.bin',
 '/home/dnth/Desktop/rag-datakit/nbs/wandb/run-20250908_143209-7d4hqku2/files/models/cxs-modernbert-embed-base/tokenizer.json',
 '/home/dnth/Desktop/rag-datakit/nbs/wandb/run-20250908_143209-7d4hqku2/files/models/cxs-modernbert-embed-base/README.md',
 '/home/dnth/Desktop/rag-datakit/nbs/wandb/run-20250908_143209-7d4hqku2/files/models/cxs-modernbert-embed-ba

## Training Results and Next Steps

The training completed successfully with a final validation loss of 0.00813, showing that our model has learned to effectively distinguish between semantically similar and dissimilar job descriptions. The Weights & Biases dashboard provides detailed metrics and visualizations of the training process.

### Next Steps

To use this model in production:

1. **Load the model** using `SentenceTransformer('./models/all-minilm-l6-v2')`
2. **Evaluate** on a test set to verify performance on unseen data
3. **Deploy** in your RAG pipeline for improved job description matching
4. **Publish** to the Hugging Face Hub (uncomment the last cell) to share with the community

The fine-tuned model is now ready to provide more accurate semantic similarity scores for job descriptions in your retrieval-augmented generation workflows.

In [10]:
wandb.finish()

eval/loss,█▃▂▁▁
eval/runtime,█▁▂▁▁
eval/samples_per_second,▁█▇██
eval/steps_per_second,▁█▇██
train/epoch,▁▁▃▃▅▅▆▆███
train/global_step,▁▁▃▃▅▅▆▆███
train/grad_norm,█▃▃▁▁
train/learning_rate,█▇▄▂▁
train/loss,█▁▁▁▁
eval/loss,0.00738
eval/runtime,5.0741


In [11]:
trainer.model.push_to_hub("dnth/ssf-retriever-modernbert-embed-base", exist_ok=True)

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /tmp/tmp5mrq7gyd/tokenizer.json       :   1%|1         | 44.8kB / 3.58MB            

  /tmp/tmp5mrq7gyd/model.safetensors    :   0%|          |  553kB /  596MB            

'https://huggingface.co/dnth/ssf-retriever-modernbert-embed-base/commit/5734ff06899cfb86330da4044cca5dec7d827e40'